# EMD / CMIP-LD Development Notebook

Testing the ReportBuilder pipeline — no external CV config needed.
The builder infers CV field graphs from the expanded folder items themselves.

In [11]:
import sys
# sys.path.insert(0, '/Users/daniel.ellis/WIPwork/CMIP-LD')
# sys.path.insert(0, '/Users/daniel.ellis/WIPwork/Essential-Model-Documentation')

import importlib, json, pprint
pp = pprint.PrettyPrinter(indent=2).pprint

TEST_ITEM = {
    'validation_key': 'tempgrid_wolfiex-1777283038',
    'ui_label': 'Horizontal grid cell with a hierarchical discrete global grid grid type and 1 x 1 km resolution.',
    'description': 'TEST TEST TEST',
    'alias': ['item2', ' item3'],
    'grid_mapping': 'lambert-azimuthal-equal-area',
    'grid_type': 'hierarchical-discrete-global-grid',
    'n_cells': 1,
    'region': ['southern-hemisphere'],
    'southernmost_latitude': 1,
    'temporal_refinement': 'dynamically-stretched',
    'truncation_method': 'triangular',
    'truncation_number': 1,
    'units': 'km',
    'westernmost_longitude': 1,
    'x_resolution': 1,
    'y_resolution': 1,
    '@context': '_context',
    '@type': ['emd', 'wcrp:horizontal_grid_cell', 'esgvoc:HorizontalGridCell'],
    '@id': 'tempgrid-wolfiex-1777283038'
}
print('Test item loaded.')

Test item loaded.


## 1. Load Folder Graph & Inspect Expanded Items
At depth=4, folder items have CV field values expanded to full `@id` URIs.
This is where the CV graph URLs are inferred from.

In [12]:
import cmipld
import cmipld.utils.similarity.report_builder as rb_mod
importlib.reload(rb_mod)

from cmipld.utils.similarity.graph_loader import GraphLoader

folder_url = 'emd:horizontal_grid_cell'
print(f'Loading {folder_url}/_graph.json at depth=4 ...')
with cmipld.ensure_remote():
    data = cmipld.expand(f'{folder_url}/_graph.json', depth=4)

loader = GraphLoader(folder_url, graph_data=data)
folder_items = loader.items
print(f'Loaded {len(folder_items)} folder items\n')

# Show linked fields in the first item
print('Linked fields in first item (these tell us which fields are CV fields):')
for k, v in folder_items[0].items():
    if isinstance(v, dict) and '@id' in v:
        print(f'  {k}: {v["@id"]}')

Loading emd:horizontal_grid_cell/_graph.json at depth=4 ...
[Cache HIT] emd:horizontal_grid_cell/_graph.json (depth=4)
Loaded 19 folder items

Linked fields in first item (these tell us which fields are CV fields):
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/grid_mapping: https://constants.mipcvs.dev/grid_mapping/latitude-longitude
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/grid_type: https://constants.mipcvs.dev/grid_type/regular-latitude-longitude
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/region: https://constants.mipcvs.dev/region/global
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/temporal_refinement: https://constants.mipcvs.dev/temporal_refinement/static
  https://emd.mipcvs.dev/docs/contents/HorizontalGridCell/truncation_method: https://constants.mipcvs.dev/truncation_method/


## 2. Infer CV Graph URLs from Folder Items
`_infer_cv_graphs_from_folder` scans the existing items and derives the graph URL
from the directory part of each `@id` URI.

In [13]:
from cmipld.utils.similarity.report_builder import _infer_cv_graphs_from_folder

field_graphs = _infer_cv_graphs_from_folder(folder_items)
print(f'Inferred {len(field_graphs)} CV field graph(s):')
for field, url in field_graphs.items():
    print(f'  {field}: {url}')

Inferred 6 CV field graph(s):
  grid_mapping: https://constants.mipcvs.dev/grid_mapping/_graph.json
  grid_type: https://constants.mipcvs.dev/grid_type/_graph.json
  region: https://constants.mipcvs.dev/region/_graph.json
  temporal_refinement: https://constants.mipcvs.dev/temporal_refinement/_graph.json
  units: https://constants.mipcvs.dev/units/_graph.json
  truncation_method: https://constants.mipcvs.dev/truncation_method/_graph.json


## 3. Fetch Each CV Graph & Check Valid Values
`_fetch_cv_graph` retrieves the full list of valid values for a field.

In [14]:
from cmipld.utils.similarity.report_builder import _fetch_cv_graph

for field, graph_url in field_graphs.items():
    lookup = _fetch_cv_graph(graph_url)
    submitted = TEST_ITEM.get(field, '')
    if isinstance(submitted, list): submitted = submitted[0] if submitted else ''
    # Try probes
    probes = [submitted, submitted.lower(), submitted.replace('-','_'), submitted.replace('_','-')]
    resolved_uri = next((lookup[p] for p in probes if p in lookup), None)
    status = '✅' if resolved_uri else '❌'
    print(f'{status} {field}: {submitted!r}')
    if resolved_uri:
        print(f'    → {resolved_uri}')
    else:
        available = sorted({v.split("/")[-1] for v in lookup.values()})[:5]
        print(f'    Valid values (sample): {available}')

[Cache HIT] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
❌ grid_mapping: 'lambert-azimuthal-equal-area'
    Valid values (sample): []
[Cache HIT] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
❌ grid_type: 'hierarchical-discrete-global-grid'
    Valid values (sample): []
[Cache HIT] https://constants.mipcvs.dev/region/_graph.json (depth=2)
❌ region: 'southern-hemisphere'
    Valid values (sample): []
[Cache HIT] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
❌ temporal_refinement: 'dynamically-stretched'
    Valid values (sample): []
[Cache HIT] https://constants.mipcvs.dev/units/_graph.json (depth=2)
❌ units: 'km'
    Valid values (sample): []
[Cache HIT] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)
❌ truncation_method: 'triangular'
    Valid values (sample): []


## 4. Test `_build_links_from_folder` — Self-Contained
Returns `(field_links, field_graphs)` — no external config needed.

In [15]:
from cmipld.utils.similarity.report_builder import _build_links_from_folder

field_links, field_graphs = _build_links_from_folder(TEST_ITEM, folder_items)

print(f'{len(field_links)}/{len(field_graphs)} links resolved:\n')
for field, url in field_graphs.items():
    val = TEST_ITEM.get(field, '')
    if isinstance(val, list): val = val[0] if val else ''
    if field in field_links:
        print(f'  ✅ {field}: {val!r} → {field_links[field].split("/")[-1]}')
    else:
        print(f'  ❌ {field}: {val!r} — not found in graph')

[Cache HIT] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/region/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/units/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)
2/6 links resolved:

  ❌ grid_mapping: 'lambert-azimuthal-equal-area' — not found in graph
  ✅ grid_type: 'hierarchical-discrete-global-grid' → hierarchical-discrete-global-grid
  ❌ region: 'southern-hemisphere' — not found in graph
  ❌ temporal_refinement: 'dynamically-stretched' — not found in graph
  ❌ units: 'km' — not found in graph
  ✅ truncation_method: 'triangular' → triangular


In [ ]:
cmipld.get

## 5. Fraction Logic
Replicate what `_link_section` will display.

In [16]:
from cmipld.utils.similarity.report_builder import _is_report_skip

total_cv = sum(
    1 for k, v in TEST_ITEM.items()
    if k in field_graphs
    and not _is_report_skip(k)
    and v not in ('', None, [], {})
)
resolved = len(field_links)
fraction = f'{resolved}/{total_cv}'
pct = f'{resolved / total_cv * 100:.0f}%' if total_cv else '—'

print(f'Display: CV links resolved: {fraction} ({pct})')

Display: CV links resolved: 2/6 (33%)


## 6. Full ReportBuilder Run — No Config Needed

In [17]:
importlib.reload(rb_mod)
from cmipld.utils.similarity.report_builder import ReportBuilder

report = ReportBuilder(
    folder_url='emd:horizontal_grid_cell',
    kind='horizontal_grid_cell',
    item=TEST_ITEM,
    link_threshold=85.0,
).build()

print(f'Report generated: {len(report)} chars')

[Cache HIT] emd:horizontal_grid_cell/_graph.json (depth=4)
[Cache HIT] https://constants.mipcvs.dev/grid_mapping/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/grid_type/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/region/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/temporal_refinement/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/units/_graph.json (depth=2)
[Cache HIT] https://constants.mipcvs.dev/truncation_method/_graph.json (depth=2)


ValueError: not enough values to unpack (expected 4, got 2)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## 7. Reload & Rerun After Code Changes
Use this cell to quickly re-test after editing report_builder.py.

In [ ]:
importlib.reload(rb_mod)
from cmipld.utils.similarity.report_builder import (
    ReportBuilder, _build_links_from_folder,
    _infer_cv_graphs_from_folder, _fetch_cv_graph,
)

field_links, field_graphs = _build_links_from_folder(TEST_ITEM, folder_items)
print(f'Links: {len(field_links)}/{len(field_graphs)}')
for k, v in field_links.items():
    print(f'  {k}: {v.split("/")[-1]}')

In [20]:
!mamba install pandas --yes



                  __    __    __    __
                 /  \  /  \  /  \  /  \
                /    \/    \/    \/    \
███████████████/  /██/  /██/  /██/  /████████████████████████
              /  / \   / \   / \   / \  \____
             /  /   \_/   \_/   \_/   \    o \__,
            / _/                       \_____/  `
            |/
        ███╗   ███╗ █████╗ ███╗   ███╗██████╗  █████╗
        ████╗ ████║██╔══██╗████╗ ████║██╔══██╗██╔══██╗
        ██╔████╔██║███████║██╔████╔██║██████╔╝███████║
        ██║╚██╔╝██║██╔══██║██║╚██╔╝██║██╔══██╗██╔══██║
        ██║ ╚═╝ ██║██║  ██║██║ ╚═╝ ██║██████╔╝██║  ██║
        ╚═╝     ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚═════╝ ╚═╝  ╚═╝

        mamba (1.0.0) supported by @QuantStack

        GitHub:  https://github.com/mamba-org/mamba
        Twitter: https://twitter.com/QuantStack

█████████████████████████████████████████████████████████████


Looking for: ['pandas']

conda-forge/osx-arm64                                       Using cache
conda-forge/noar

In [ ]:
# import yaml
# import glob
# from openpyxl import Workbook
# from openpyxl.worksheet.datavalidation import DataValidation

# yaml_files = glob.glob('/Users/daniel.ellis/WIPwork/Essential-Model-Documentation/.github/ISSUE_TEMPLATE/*.yml')

# for yaml_file in yaml_files:
#     print(f"Processing {yaml_file}...")
    
#     if 'config' in yaml_file:
#         print("  Skipping config file.")
#         continue

#     with open(yaml_file, "r", encoding="utf-8") as f:
#         form = yaml.safe_load(f)

#     wb = Workbook()
#     ws = wb.active
#     ws.title = "Form"

#     fields = []
#     dropdowns = {}

#     # Extract fields
#     for item in form.get("body", []):
#         if "id" in item:
#             field_id = item["id"]
#             label = item.get("attributes", {}).get("label", field_id)
#             options = item.get("attributes", {}).get("options", [])

#             fields.append(label)

#             if options:
#                 dropdowns[label] = options

#     # Write header row
#     for col, field in enumerate(fields, start=1):
#         ws.cell(row=1, column=col, value=field)

#     # Add dropdowns (apply to first 100 rows for usability)
#     for col, field in enumerate(fields, start=1):
#         if field in dropdowns:
#             options = dropdowns[field]

#             dv = DataValidation(
#                 type="list",
#                 formula1=f'"{",".join(options)}"',
#                 allow_blank=True
#             )

#             ws.add_data_validation(dv)

#             # Apply dropdown to rows 2–100
#             col_letter = ws.cell(row=1, column=col).column_letter
#             dv.add(f"{col_letter}2:{col_letter}100")

#     # Save file
#     output_file = yaml_file.split('/')[-1].replace(".yml", ".xlsx")
#     wb.save(output_file)

#     print(f"  Created {output_file}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

: 